# Quick Start: Running a Feed Forward Model on GIFT-Eval


This notebook demonstrates how to run a GluonTS [Simple Feed Forward model](https://ts.gluon.ai/dev/api/gluonts/gluonts.torch.model.simple_feedforward.html) on the GIFT-Eval benchmark.


## Dataset


First, let's load the dataset. For the sake of brevity, we'll only load two datasets.


In [ ]:
import json
from dotenv import load_dotenv
from datetime import datetime

format = "%m/%d/%Y %I:%M:%S%p"


def print_timestamp():
    now = datetime.now()
    formatted_time = now.strftime(format)
    print(f"Last time ran: {formatted_time}")


# Load environment variables
load_dotenv()

# Create a set of all the short dataset names
short_dataset_names = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
short_datasets = set(short_dataset_names.split())

# Name of the short dataset we'll use
short_dataset = "m4_hourly"

# Create a set of all the medium to long dataset names
med_long_dataset_names = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
med_long_datasets = set(med_long_dataset_names.split())

# Name of the medium to long dataset we'll use
med_long_dataset = "bizitobs_l2c/H"

# Combine the datasets into one list
all_datasets = [short_dataset, med_long_dataset]

# Load the dataset properties map
dataset_properties_map = json.load(open("dataset_properties.json"))

print_timestamp()

Finished running at 03/02/2025 04:56:10PM


## Training


Initialize a CSV file to save our model's performance on each dataset


In [27]:
import csv
import os

# Name of the directory where our model's results will be saved
output_directory = "./results/feedforward"

# Ensure the directory exists
os.makedirs(output_directory, exist_ok=True)

# Define the CSV file's path
csv_file_path = os.path.join(output_directory, "results.csv")

# Initialize the CSV file's header
with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)

    # Write the header
    writer.writerow(
        [
            "dataset",
            "model",
            "eval_metrics/MSE[mean]",
            "eval_metrics/MSE[0.5]",
            "eval_metrics/MAE[0.5]",
            "eval_metrics/MASE[0.5]",
            "eval_metrics/MAPE[0.5]",
            "eval_metrics/sMAPE[0.5]",
            "eval_metrics/MSIS",
            "eval_metrics/RMSE[mean]",
            "eval_metrics/NRMSE[mean]",
            "eval_metrics/ND[0.5]",
            "eval_metrics/mean_weighted_sum_quantile_loss",
            "domain",
            "num_variates",
        ]
    )

print_timestamp()

Finished running at 03/02/2025 04:56:10PM


Load the metrics we'll evaluate our model on


In [28]:
from gluonts.ev.metrics import (
    MSE,
    MAE,
    MASE,
    MAPE,
    SMAPE,
    MSIS,
    RMSE,
    NRMSE,
    ND,
    MeanWeightedSumQuantileLoss,
)

# Instantiate metrics
metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(
        quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    ),
]

print_timestamp()

Finished running at 03/02/2025 04:56:10PM


Before training and evaluation, let's define some helper functions


In [29]:
from gluonts.model import evaluate_model
from gluonts.torch.model.simple_feedforward import SimpleFeedForwardEstimator
from gluonts.time_feature import get_seasonality
from gift_eval.data import Dataset

terms = ["short", "medium", "long"]


def get_dataset_key(dataset_name):
    return dataset_name.split("/")[0] if "/" in dataset_name else dataset_name.lower()


def get_dataset_freq(dataset_name):
    key = get_dataset_key(dataset_name)
    return (
        dataset_name.split("/")[1]
        if "/" in dataset_name
        else dataset_properties_map[key]["frequency"]
    )


def get_dataset_config(dataset_name, term):
    key = get_dataset_key(dataset_name)
    freq = get_dataset_freq(dataset_name)
    return f"{key}/{freq}/{term}"


def write_to_csv(csv_file_path, dataset_name, term, results):
    # Initialize dataset's configuation
    dataset_config = get_dataset_config(dataset_name, term)

    # Write the results to the CSV file
    with open(csv_file_path, "a", newline="") as csvfile:
        key = get_dataset_key(dataset_name)

        writer = csv.writer(csvfile)
        writer.writerow(
            [
                dataset_config,
                "feedforward",
                results["MSE[mean]"][0],
                results["MSE[0.5]"][0],
                results["MAE[0.5]"][0],
                results["MASE[0.5]"][0],
                results["MAPE[0.5]"][0],
                results["sMAPE[0.5]"][0],
                results["MSIS"][0],
                results["RMSE[mean]"][0],
                results["NRMSE[mean]"][0],
                results["ND[0.5]"][0],
                results["mean_weighted_sum_quantile_loss"][0],
                dataset_properties_map[key]["domain"],
                dataset_properties_map[key]["num_variates"],
            ]
        )


def check_if_multivariate(dataset_name, term):
    # Get the dataset's target dimension
    target_dimension = Dataset(
        name=dataset_name,
        term=term,
        to_univariate=False,
    ).target_dim

    # Check if the dataset is already univariate
    return target_dimension > 1


def train_eval(dataset_name, term, csv_file_path):
    # Check if the dataset is multivariate
    is_multivariate = check_if_multivariate(dataset_name, term)

    # True if the dataset is multivariate
    to_univariate = True if is_multivariate else False

    # Initialize dataset
    dataset = Dataset(name=dataset_name, term=term, to_univariate=to_univariate)

    # Get the seasonal component's length
    season_length = get_seasonality(dataset.freq)

    # Get the dataset's prediction length
    prediction_length = dataset.prediction_length

    # Define hyperparameters
    trainer_kwargs = {"max_epochs": 1}

    # Instantiate the model
    estimator = SimpleFeedForwardEstimator(
        prediction_length=prediction_length,
        context_length=prediction_length,
        trainer_kwargs=trainer_kwargs,
    )

    # Train the model on the validation set
    predictor = estimator.train(dataset.validation_dataset)

    # Evaluate the model on the test set
    results = evaluate_model(
        predictor,
        test_data=dataset.test_data,
        metrics=metrics,
        batch_size=512,
        axis=None,
        mask_invalid_label=True,
        allow_nan_forecast=False,
        seasonality=season_length,
    )

    write_to_csv(csv_file_path, dataset_name, term, results)

    print(f"Results for {dataset_name} have been written to {csv_file_path}")

Now, we'll train and evaluate the model on each dataset


In [30]:
# Train and evaluate the model on all of the datasets
for dataset_name in all_datasets:
    print(f"Processing dataset: {dataset_name}")

    for term in terms:
        if term != "short" and dataset_name not in med_long_datasets:
            continue
        train_eval(dataset_name, term, csv_file_path)

print_timestamp()

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\Mike\anaconda3\envs\gift\Lib\site-packages\lightning\pytorch\trainer\configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.

  | Name  | Type                   | Params | Mode 
---------------------------------------------------------
0 | model | SimpleFeedForwardModel | 21.2 K | train
---------------------------------------------------------
21.2 K    Trainable params
0         Non-trainable params
21.2 K    Total params
0.085     Total estimated model params size (MB)
11        Modules in train mode
0         Modules in eval mode


Processing dataset: m4_hourly
Epoch 0: |          | 50/? [00:00<00:00, 93.69it/s, v_num=4, train_loss=6.180]

Epoch 0, global step 50: 'train_loss' reached 6.18089 (best 6.18089), saving model to 'c:\\Users\\Mike\\vscode\\gift-eval\\notebooks\\lightning_logs\\version_4\\checkpoints\\epoch=0-step=50.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: |          | 50/? [00:00<00:00, 91.51it/s, v_num=4, train_loss=6.180]


414it [00:07, 56.59it/s]
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name  | Type                   | Params | Mode 
---------------------------------------------------------
0 | model | SimpleFeedForwardModel | 21.2 K | train
---------------------------------------------------------
21.2 K    Trainable params
0         Non-trainable params
21.2 K    Total params
0.085     Total estimated model params size (MB)
11        Modules in train mode
0         Modules in eval mode


...

Results for bizitobs_l2c/H have been written to ./results/feedforward\results.csv
Epoch 0: |          | 50/? [00:00<00:00, 78.56it/s, v_num=7, train_loss=4.760]

Epoch 0, global step 50: 'train_loss' reached 4.76362 (best 4.76362), saving model to 'c:\\Users\\Mike\\vscode\\gift-eval\\notebooks\\lightning_logs\\version_7\\checkpoints\\epoch=0-step=50.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: |          | 50/? [00:00<00:00, 76.66it/s, v_num=7, train_loss=4.760]


7it [00:00, 19.07it/s]

Results for bizitobs_l2c/H have been written to ./results/feedforward\results.csv
Finished running at 03/02/2025 04:56:21PM


## Results

Running the above cell will generate a csv file called `all_results.csv` under the `results/feedforward` folder containing the results for the Feed Forward model on the gift-eval benchmark. The csv file will look like this:


In [31]:
import pandas as pd

print_timestamp()
df = pd.read_csv("./results/feedforward/results.csv")
df.head()

Finished running at 03/02/2025 04:56:21PM


,dataset,model,eval_metrics/MSE[mean],eval_metrics/MSE[0.5],eval_metrics/MAE[0.5],eval_metrics/MASE[0.5],eval_metrics/MAPE[0.5],eval_metrics/sMAPE[0.5],eval_metrics/MSIS,eval_metrics/RMSE[mean],eval_metrics/NRMSE[mean],eval_metrics/ND[0.5],eval_metrics/mean_weighted_sum_quantile_loss,domain,num_variates
0,m4_hourly/H/short,feedforward,4.196692e+07,4.196692e+07,1040.483494,10.093299,0.891260,0.323162,184.532121,6478.188029,0.884416,0.142049,0.148502,Econ/Fin,1
1,bizitobs_l2c/H/short,feedforward,2.263157e+02,2.263157e+02,11.569043,1.114044,1.152366,0.973152,13.557010,15.043794,0.810899,0.623601,0.510077,Web/CloudOps,7
2,bizitobs_l2c/H/medium,feedforward,8.650632e+01,8.650632e+01,6.117985,0.623422,0.709238,0.797339,5.678183,9.300878,0.563182,0.370453,0.303264,Web/CloudOps,7
3,bizitobs_l2c/H/long,feedforward,1.032249e+02,1.032249e+02,6.825742,0.720265,0.801138,0.825184,7.381830,10.159964,0.620600,0.416936,0.343918,Web/CloudOps,7
